# Stage 1 — candidate search, ALL runs in one go
Order: test main → test extra → train main → train extra (~6 h total). Each step saves its own file, so a late failure keeps earlier results.

Inputs: competition data + `er-code`. Session: **CPU**, Internet **On**. Run with **Save Version → Save & Run All (Commit)**.

In [ ]:
!pip install -q anyascii==0.3.3 sparse_dot_topn==1.2.0

In [ ]:
import os, shutil

def find(filename, marker):
    """Walk /kaggle/input (following symlinks) and return the folder above `marker`."""
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        if filename in files and root.replace("\\", "/").endswith(marker):
            return root[: -len(marker)].rstrip("/")
    return None

code_src = find("normalize.py", "/er")
DATA_DIR = find("train_source1.tsv", "/train")
if DATA_DIR is None:   # fall back to the path that worked in the interactive notebook
    p = "/kaggle/input/datasets/mithra009/amc-2026/student_resource/dataset"
    DATA_DIR = p if os.path.exists(p + "/train/train_source1.tsv") else None

if code_src is None or DATA_DIR is None:
    print("Attached inputs:")
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        depth = root.count("/") - 2
        if depth <= 5:
            print("  " * depth + root.rsplit("/", 1)[-1] + "/", files[:6])
    raise SystemExit(f"MISSING INPUT -> code found: {code_src is not None}, data found: {DATA_DIR is not None}. "
                     "Add BOTH the competition dataset (amc-2026) and er-code via 'Add Input'.")

shutil.rmtree("/kaggle/working/src", ignore_errors=True)
shutil.copytree(code_src, "/kaggle/working/src")
os.environ["DATA_DIR"] = DATA_DIR
print("code:", code_src)
print("data:", DATA_DIR)
print(sorted(os.listdir(DATA_DIR + "/train")), sorted(os.listdir(DATA_DIR + "/test")))

In [ ]:
%%writefile /kaggle/working/src/er/candidates.py
"""Multi-pass sparse TF-IDF candidate generation.

Direction: every S2/S3 record queries the S1 records of the same country and keeps its
top-k S1 neighbours per pass. (Each S2/S3 record belongs to at most one S1, so this is the
natural direction; per-S1 candidate lists are the inverse of these edges.)
Common tokens (document frequency > max_df) are dropped: little TF-IDF weight, most of the cost.
Identical query strings are searched once and the result is shared.
Licenses: scikit-learn (BSD-3), sparse_dot_topn (Apache-2.0), scipy/numpy (BSD).
"""
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
from .data import log

# id, name, field, analyzer, ngram_range, max_df, k (neighbours kept)
PASSES = [
    (0, "na_w", "na", "word", (1, 1), 0.02, 5),
    (1, "ad_w", "ad", "word", (1, 2), 0.01, 5),
    (2, "nm_w", "nm", "word", (1, 1), 0.02, 1),
]
# Extra passes, run only on "hard" queries (non-Latin-script name, or empty/very short address):
# these are where the main passes miss most, and they are a small share of all records.
EXTRA_PASSES = [
    (3, "nm_w10", "nm", "word",    (1, 1), 0.02, 10),
    (4, "sk_c3",  "sk", "char_wb", (3, 3), 0.02, 5),
]
PASS_NAMES = {p[0]: p[1] for p in PASSES + EXTRA_PASSES}

def hard_query_mask(s23):
    """Non-Latin-script name (transliterated) OR address with <= 3 tokens after cleaning."""
    raw = s23.business_name.values
    nonlatin = np.fromiter((any(ord(ch) > 0x2FF and ch.isalpha() for ch in x) for x in raw), bool, len(raw))
    short_addr = s23.ad.str.count(" ").values + (s23.ad.values != "") <= 3
    return nonlatin | short_addr

def run_pass(s1, s23, q_idx, spec, n_threads, countries=None, k_override=None, chunk=200_000):
    """Returns DataFrame(q, s1, score, rank, pass) with global row indices into s23 / s1."""
    pid, name, field, analyzer, ngram, max_df, k = spec
    k = k_override or k
    out = []
    cs = sorted(set(s1.country.unique()) & set(s23.country.unique()))
    if countries:
        cs = [c for c in cs if c in countries]
    for c in cs:
        a_idx = np.flatnonzero(s1.country.values == c)
        b_all = np.flatnonzero(s23.country.values == c)
        b_idx = np.intersect1d(b_all, q_idx, assume_unique=True)
        if len(a_idx) == 0 or len(b_idx) == 0:
            continue
        t = time.time()
        vec = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram, max_df=max_df, min_df=1,
                              sublinear_tf=True, dtype=np.float32, lowercase=False)
        vec.fit(pd.concat([s1[field].iloc[a_idx], s23[field].iloc[b_all]], ignore_index=True))
        A_T = vec.transform(s1[field].iloc[a_idx]).T.tocsr()
        t_fit = time.time() - t; t = time.time()
        # search each distinct query string once
        codes, uniq = pd.factorize(s23[field].values[b_idx])
        res_q, res_s1, res_sc, res_rk = [], [], [], []
        for i in range(0, len(uniq), chunk):
            B = vec.transform(uniq[i:i + chunk])
            C = sp_matmul_topn(B, A_T, top_n=k, threshold=0.0, sort=True, n_threads=n_threads)
            rows = np.repeat(np.arange(C.shape[0]), np.diff(C.indptr))
            res_q.append(rows + i); res_s1.append(C.indices); res_sc.append(C.data)
            res_rk.append(np.arange(C.nnz) - C.indptr[rows])
        uq = np.concatenate(res_q); us1 = np.concatenate(res_s1)
        usc = np.concatenate(res_sc); urk = np.concatenate(res_rk)
        # expand unique-string results back to every query row (vectorized join on string id)
        res = pd.DataFrame({"u": uq, "s1": a_idx[us1].astype(np.int32), "score": usc.astype(np.float32),
                            "rank": urk.astype(np.int8)})
        qdf = pd.DataFrame({"u": codes, "q": b_idx.astype(np.int32)})
        out.append(qdf.merge(res, on="u", how="inner")[["q", "s1", "score", "rank"]])
        del res, qdf
        log(f"  pass {name:5} country={c:8} S1={len(a_idx):>9,} queries={len(b_idx):>9,} "
            f"(unique {len(uniq):>9,}) vocab={len(vec.vocabulary_):>9,} fit {t_fit:5.0f}s  search {time.time()-t:5.0f}s")
        del A_T, vec
    df = pd.concat(out, ignore_index=True) if out else pd.DataFrame(
        {"q": np.array([], np.int32), "s1": np.array([], np.int32), "score": np.array([], np.float32), "rank": np.array([], np.int8)})
    df["pass"] = np.int8(pid)
    return df

def union(edges, k=None):
    """Union of edges (optionally rank < k) -> unique (q, s1) pairs."""
    e = edges if k is None else edges[edges["rank"] < k]
    return e[["q", "s1"]].drop_duplicates()


In [ ]:
%%writefile /kaggle/working/src/run_stage1.py
#!/usr/bin/env python3
"""Stage 1 — candidate generation (+ recall report on train).

Full run:
    python run_stage1.py --data-dir DATASET_DIR --split test
    python run_stage1.py --data-dir DATASET_DIR --split train
Split across two notebooks by country (each writes its own edges file):
    ... --countries US
    ... --countries India France
Quick recall estimate on a random fraction of queries:
    ... --split train --query-frac 0.05
"""
import argparse, os, sys, time
import numpy as np
import pandas as pd
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from er.data import load_split, truth_index, log
from er.candidates import PASSES, EXTRA_PASSES, PASS_NAMES, hard_query_mask, run_pass, union

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", required=True, help="folder containing train/ and test/")
    ap.add_argument("--split", default="train", choices=["train", "test"])
    ap.add_argument("--work-dir", default="/kaggle/working/er_work")
    ap.add_argument("--query-frac", type=float, default=1.0)
    ap.add_argument("--countries", nargs="*", default=None, help="restrict to these country labels")
    ap.add_argument("--threads", type=int, default=os.cpu_count())
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--extra", action="store_true", help="run only the extra passes on hard queries")
    a = ap.parse_args()
    os.makedirs(a.work_dir, exist_ok=True)
    T0 = time.time()

    s1, s23, gt = load_split(a.data_dir, a.split, a.work_dir, n_proc=a.threads)
    log(f"S1={len(s1):,}  S2+S3={len(s23):,}")
    log(f"countries S1={s1.country.value_counts().to_dict()}  S2+S3={s23.country.value_counts().to_dict()}")
    rng = np.random.default_rng(a.seed)
    q_idx = np.sort(rng.choice(len(s23), size=int(len(s23) * a.query_frac), replace=False)) if a.query_frac < 1 \
        else np.arange(len(s23))
    if a.countries:
        q_idx = q_idx[np.isin(s23.country.values[q_idx], a.countries)]
    passes = PASSES
    if a.extra:
        hard = hard_query_mask(s23)
        q_idx = q_idx[hard[q_idx]]
        passes = EXTRA_PASSES
        log(f"EXTRA mode: {hard.mean():.1%} of S2/S3 records are hard queries")
    log(f"querying {len(q_idx):,} S2/S3 records, countries={a.countries or 'all'}, threads={a.threads}")

    edges, times = [], {}
    for spec in passes:
        t = time.time()
        edges.append(run_pass(s1, s23, q_idx, spec, a.threads, countries=a.countries))
        times[spec[1]] = time.time() - t
    E = pd.concat(edges, ignore_index=True)
    tag = f"{a.split}_q{a.query_frac:g}" + ("_extra" if a.extra else "") + (f"_{'-'.join(a.countries)}" if a.countries else "")
    path = os.path.join(a.work_dir, f"edges_{tag}.parquet")
    E.to_parquet(path, index=False)
    log(f"saved {path} ({len(E):,} rows)")

    print("\n================ STAGE 1 REPORT ================")
    print(f"split={a.split} query_frac={a.query_frac} countries={a.countries or 'all'} total_time={(time.time()-T0)/60:.1f} min")
    for p, s in times.items():
        print(f"  pass {p:5}: {s/60:6.1f} min")
    u = union(E)
    print(f"  unique candidate pairs: {len(u):,}  ({len(u)/max(len(q_idx),1):.1f} per S2/S3 record)")
    n_s1_cands = u.s1.nunique()
    print(f"  S1 entities with >=1 candidate: {n_s1_cands:,} of {len(s1):,}")
    if gt is not None:
        owner = truth_index(gt, s1, s23)
        n_true = int((owner[q_idx] >= 0).sum())
        hit = (owner[u.q.values] == u.s1.values).sum()
        print(f"  recall (true S1 among candidates): {hit/n_true:.4f}  ({hit:,} of {n_true:,})")
        for pid in sorted(E["pass"].unique()):
            name = PASS_NAMES[int(pid)]; e = E[E["pass"] == pid]
            print(f"    {name:5}: {(owner[e.q.values] == e.s1.values).sum()/n_true:.4f}")
    print("================================================")

if __name__ == "__main__":
    main()


## 1. Test — main passes

In [ ]:
!python /kaggle/working/src/run_stage1.py --data-dir "$DATA_DIR" --split test

## 2. Test — extra passes on hard records

In [ ]:
!python /kaggle/working/src/run_stage1.py --data-dir "$DATA_DIR" --split test --extra

## 3. Train — main passes

In [ ]:
!python /kaggle/working/src/run_stage1.py --data-dir "$DATA_DIR" --split train

## 4. Train — extra passes on hard records

In [ ]:
!python /kaggle/working/src/run_stage1.py --data-dir "$DATA_DIR" --split train --extra

In [ ]:
!ls -la /kaggle/working/er_work && df -h /kaggle/working